# LightGBM Model Development

### [1] 라이브러리불러오기

In [ ]:
%pip install lightgbm

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import re

from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import train_test_split

# =========================
# sklearn 버전에 따라 StratifiedGroupKFold가 없을 수도 있어서 예외 처리
# =========================
try:
    from sklearn.model_selection import StratifiedGroupKFold
    USE_STRATIFIED_GROUP = True
except ImportError:
    from sklearn.model_selection import GroupKFold
    USE_STRATIFIED_GROUP = False

### [2] 데이터셋 불러오기

In [ ]:
# =========================
# azureml-core 버전 1.0.72 이상이 필요함
# 버전 1.1.34 이상의 azureml-dataprep[pandas]가 필요함.
# =========================
from azureml.core import Workspace, Dataset

subscription_id = 'YOUR_SUBSCRIPTION_ID'
resource_group = 'YOUR_RESOURCE_GROUP'
workspace_name = 'YOUR_WORKSPACE_NAME'

workspace = Workspace(subscription_id, resource_group, workspace_name)

dataset = Dataset.get_by_name(workspace, name='YOUR_DATASET')
dataset.to_pandas_dataframe()

In [ ]:
# =========================
# df로 받기
# =========================
df_origin = dataset.to_pandas_dataframe()
print(df_origin.shape)

In [ ]:
# =========================
# 컬럼 선택하기
# =========================
df_clean = df_origin[[
    'voltage',
    'current',
    'battery_temp',
    'ambient_temp',
    'delta_v',
    'delta_i',
    'temp_diff',
    'joule_heating_stress',
    'rolling_abs_power_70min',
    'thermal_stress',
    'temp_diff_mean',
    'vehicle_id',
    'current_status_label'
]]
print('===df_clean.shape===',df_clean.shape)

In [ ]:
# =========================
# 결측치 확인하기
# =========================
print(df_clean.isna().sum())

# #3. 필요하다면 결측치 처리하기
# df_clean = 
# print('===df_final_clean===', df_clean.shape)

### [3] train/test 차량이 겹치지 않도록 분리하기

In [ ]:
# df_clean에서 차량종류 확인하기
print('===차량종류===', df_clean['vehicle_id'].unique())

In [ ]:
# =========================
# train, test 의 차량 종류 다르게 나누기
# =========================

# ----------------
# 차량 목록 추출
# ----------------
vehicle_df = df_clean[['vehicle_id']].drop_duplicates().copy()

# ----------------
# 80:20 분리
# ----------------
train_vids, test_vids = train_test_split(
    vehicle_df['vehicle_id'],
    test_size=0.2,
    random_state=42
)

# ----------------
# 최종 분리
# ----------------
df_train = df_clean[df_clean['vehicle_id'].isin(train_vids)].copy()
df_test  = df_clean[df_clean['vehicle_id'].isin(test_vids)].copy()


# ----------------
# 검증
# ----------------
print(sorted(set(df_train['vehicle_id']) & set(df_test['vehicle_id'])))  
print(df_train['vehicle_id'].nunique(), df_test['vehicle_id'].nunique())
print(df_train['vehicle_id'].unique(), df_test['vehicle_id'].unique())

# 1. 모델링

### [0] 기본 설정

In [ ]:
# =========================
# 중요한 컬럼 미리 담기 
# =========================
df = df_train.copy()

# 정답 컬럼
TARGET_COL = 'current_status_label'

# 차량 이름
INDEX_COL = 'vehicle_id'

# cv 교차검증 시 교차 검증할 set 개수
N_SPLITS = 5


RANDOM_STATE = 42

# threshold 값 설정
threshold = 0.15 

df.shape

### [0-1] 특수문자 에러 방지
LightGBM은 특수문자 에러가 자주 일어남.

In [21]:

def clean_feature_name(col):
    col = str(col).strip()

    replacements = {
        '%': 'pct',
        '°C': 'degC',
        '°': 'deg',
        '[': '_',
        ']': '',
        '(': '_',
        ')': '',
        '/': '_',
        '\\': '_',
        '-': '_',
        '.': '_',
        ' ': '_',
        ',': '_',
        ':': '_',
        ';': '_',
        '{': '_',
        '}': '_',
        '"': '',
        "'": '',
        '__': '_'
    }

    for old, new in replacements.items():
        col = col.replace(old, new)

    col = re.sub(r'[^A-Za-z0-9_]', '', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')

    # 빈 문자열 방지
    if col == '':
        col = 'feature'

    # 숫자로 시작하면 접두사 추가
    if col[0].isdigit():
        col = 'f_' + col

    return col


def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")

    return new_cols


### [1] 제외할 컬럼 : 
정답컬럼, 기본키컬럼(vehicle_id,Time [s]) 상위 z-score 1위 컬럼(Z_Delta_I).

In [ ]:
# =========================
# 정답 레이블, 차량 이름 컬럼 제외
# =========================
manual_drop = [
    TARGET_COL,
    INDEX_COL
]


leakage_cols = set()

for col in df.columns:
    col_low = col.lower()

    if col in manual_drop:
        leakage_cols.add(col)
        continue

# =========================
# 최종 컬럼만 유지
# =========================
leakage_cols = sorted([c for c in leakage_cols if c in df.columns])

print("=" * 60)
print('제외할 컬럼 확인',leakage_cols[:100]) 
print("=" * 60)
print("현재 train shape:", df.shape)

### [2] X / y / groups 구성

In [ ]:
y = df[TARGET_COL].copy()
groups = df[INDEX_COL].copy()

X = df.drop(columns=leakage_cols, errors='ignore').copy()

print("현재 train shape:", X.shape)
# print("원본 X 컬럼:", X.columns.tolist())
# print("최종 X shape:", X.shape)
# print("최종 feature 수:", X.shape[1])

x_selected_cols = pd.Series(X.columns).reset_index(drop=True).T
print("====남아있는 컬럼====")
print(x_selected_cols)

### [3] LightGBM용 컬럼명 정리 : 특수문자 에러 방지 

In [ ]:
original_cols = X.columns.tolist()
clean_cols = [clean_feature_name(c) for c in original_cols]
clean_cols = make_unique_columns(clean_cols)

col_map = dict(zip(original_cols, clean_cols))
reverse_col_map = dict(zip(clean_cols, original_cols))

X = X.rename(columns=col_map)

print("\n====컬럼명 변경====")
for old, new in list(col_map.items())[:30]:
    print(f"{old}  -->  {new}")

print("\n정리 후 X shape:", X.shape)

x_final_cols = pd.DataFrame({
    '컬럼명원본': original_cols,
    '변경된컬럼명': X.columns,
    'dtype': X.dtypes
}).reset_index(drop=True)

print("\n====최종 컬럼 정보====")
print(x_final_cols.head(20))

### [4] x_final 결측치 확인

In [ ]:
print("\n====결측치====")
print(X.isna().sum().sort_values(ascending=False).head(20))


### [5] CV splitter 

In [ ]:
if USE_STRATIFIED_GROUP:
    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )
    split_iter = cv.split(X, y, groups)
    print("\nStratifiedGroupKFold 사용")
else:
    cv = GroupKFold(n_splits=N_SPLITS)
    split_iter = cv.split(X, y, groups)
    print("\nGroupKFold 사용 (sklearn 버전 확인 필요)")


In [ ]:
oof_pred = np.zeros(len(df), dtype=int)
fold_scores = []
feature_importance_list = []

for fold, (tr_idx, va_idx) in enumerate(split_iter, 1):
    X_train, X_valid = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_train, y_valid = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()

    train_groups = groups.iloc[tr_idx]
    valid_groups = groups.iloc[va_idx]

    overlap = set(train_groups) & set(valid_groups)

    print(f"\n{'='*70}")
    print(f"Fold {fold}")
    print(f"train rows: {len(tr_idx):,}, valid rows: {len(va_idx):,}")
    print(f"train vehicle 수: {train_groups.nunique()}, valid vehicle 수: {valid_groups.nunique()}")
    print(f"겹치는 vehicle_id 수: {len(overlap)}")

    print("\ntrain label 비율")
    print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

    print("\nvalid label 비율")
    print((y_valid.value_counts(normalize=True).sort_index() * 100).round(2))

    model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=y.nunique(),
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    proba = model.predict_proba(X_valid)
    classes = model.classes_
    DANGER_IDX = np.where(classes == 2)[0][0]   # danger 라벨이 2라는 가정
    DANGER_THRESHOLD = 0.15

    pred = []
    for p in proba:
        if p[DANGER_IDX] >= DANGER_THRESHOLD:
            pred.append(classes[DANGER_IDX])
        else:
            pred.append(classes[int(np.argmax(p))])

    pred = np.array(pred)
    oof_pred[va_idx] = pred

    fold_f1 = f1_score(y_valid, pred, average='macro')
    fold_scores.append(fold_f1)

    print(f"\nFold {fold} Macro F1: {fold_f1:.4f}")

    fi = pd.DataFrame({
        'feature_clean': X.columns,
        'importance': model.feature_importances_,
        'gain_importance': model.booster_.feature_importance(importance_type='gain'),
        'fold': fold
    })
    fi['feature_original'] = fi['feature_clean'].map(reverse_col_map)

    feature_importance_list.append(fi)




In [ ]:
print("\n" + "=" * 60)
print("CV 결과")
print("Fold별 Macro F1:", [round(s, 4) for s in fold_scores])
print("평균 Macro F1:", round(np.mean(fold_scores), 4))
print("표준편차:", round(np.std(fold_scores), 4))

print("\nOOF Classification Report")
print(classification_report(y, oof_pred, digits=4))

print("\nOOF Confusion Matrix")
print(confusion_matrix(y, oof_pred))

# =========================
# 평균 feature importance
# =========================
fi_all = pd.concat(feature_importance_list, ignore_index=True)

fi_mean = (
    fi_all.groupby('feature_clean', as_index=False)[['importance', 'gain_importance']]
    .mean()
    .sort_values('gain_importance', ascending=False)
)

print("\n평균 Feature Importance Top 30")
print(fi_mean.head(30))


In [ ]:
# =========================
# feature 중요도 그래프
# =========================

import matplotlib.pyplot as plt

top10 = fi_mean.head(10).sort_values('gain_importance')
top10.plot.barh(x='feature_clean', y='gain_importance', figsize=(8, 4), legend=False)

plt.title('Feature Importance Top 10')
plt.xlabel('Gain Importance')
plt.ylabel('')
plt.tight_layout()
plt.show()


### [6] train-test 최종 모델링을 위한 정보 확인

In [ ]:
final_train_df = df_train.copy()
final_test_df = df_test.copy()

X_train_final = final_train_df.drop(columns=leakage_cols, errors='ignore').copy()
y_train_final = final_train_df[TARGET_COL].copy()

X_test_final = final_test_df.drop(columns=leakage_cols, errors='ignore').copy()
y_test_final = final_test_df[TARGET_COL].copy()

# 컬럼명 정리
X_train_final = X_train_final.rename(columns=col_map)
X_test_final = X_test_final.rename(columns=col_map)

# train/test 컬럼 순서 맞추기
X_test_final = X_test_final[X_train_final.columns]

print("final train shape:", X_train_final.shape)
print("final test shape:", X_test_final.shape)
print("train vehicle 수:", final_train_df[INDEX_COL].nunique())
print("test vehicle 수:", final_test_df[INDEX_COL].nunique())
print("겹치는 vehicle 수:", len(set(final_train_df[INDEX_COL]) & set(final_test_df[INDEX_COL])))

### [7] 모델돌리기 threhold = 0.15

In [ ]:
final_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=y_train_final.nunique(),
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)

final_model.fit(X_train_final, y_train_final)

proba = final_model.predict_proba(X_test_final)
classes = final_model.classes_

# =========================
# danger 라벨 인덱스
# =========================
DANGER_IDX = np.where(classes == 2)[0][0]   # 라벨이 0/1/2일 때
DANGER_THRESHOLD = 0.15

test_pred = []
for p in proba:
    if p[DANGER_IDX] >= DANGER_THRESHOLD:
        test_pred.append(classes[DANGER_IDX])
    else:
        test_pred.append(classes[int(np.argmax(p))])

test_pred = np.array(test_pred)

print("\n" + "=" * 60)
print("Hold-out Test Classification Report")
print(classification_report(y_test_final, test_pred, digits=4))

print("\nHold-out Test Confusion Matrix")
print(confusion_matrix(y_test_final, test_pred))

print("\nHold-out Test Macro F1:", round(f1_score(y_test_final, test_pred, average='macro'), 4))

In [ ]:
print("X_train_final columns:", X_train_final.columns.tolist())
print("Does X contain target?", "current_status_label" in X_train_final.columns)
print("Does X contain bsi_label?", "bsi_label" in X_train_final.columns)
print("Does X contain received_at?", "received_at" in X_train_final.columns)